# Playing with nnInteractive

In [1]:
%cd ../..

/home/bhkuser/bhklab/katy/bhklab-nninteractive


## Dependencies, Constants, and Helper Functions

In [2]:
import os
import random
import SimpleITK as sitk
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np 

from pathlib import Path

from damply import dirs

In [3]:
from workflow.scripts.evaluate import *
from workflow.scripts.utils.visualization import pos_neg_true_visual
from workflow.scripts.utils.masks import find_first_last_slice, array_to_coords, find_max_area_slice
from workflow.scripts.utils.annotations import get_line_from_recist, get_bbox_from_line
from workflow.scripts.utils.prompts import get_prompt_points, transform_prompt_points

/home/bhkuser/bhklab/katy/bhklab-nninteractive/.pixi/envs/dev/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# from huggingface_hub import snapshot_download  # Install huggingface_hub if not already installed

# --- Download Trained Model Weights (~400MB) ---
REPO_ID = "nnInteractive/nnInteractive"
MODEL_NAME = "nnInteractive_v1.0"  # Updated models may be available in the future
DOWNLOAD_DIR = dirs.PROJECT_ROOT / "nnInteractive_test/model"  # Specify the download directory

# download_path = snapshot_download(
#     repo_id=REPO_ID,
#     allow_patterns=[f"{MODEL_NAME}/*"],
#     local_dir=DOWNLOAD_DIR
# )

# The model is now stored in DOWNLOAD_DIR/MODEL_NAME.

# --- Initialize Inference Session ---
from nnInteractive.inference.inference_session import nnInteractiveInferenceSession

# Try to make reproducible 
torch.manual_seed(42) 
random.seed(42) 
np.random.seed(42)

nnUNet_raw is not defined and nnU-Net can only be used on data for which preprocessed files are already present on your system. nnU-Net cannot be used for experiment planning and preprocessing like this. If this is not intended, please read documentation/setting_up_paths.md for information on how to set this up properly.
nnUNet_preprocessed is not defined and nnU-Net can not be used for preprocessing or training. If this is not intended, please read documentation/setting_up_paths.md for information on how to set this up.
nnUNet_results is not defined and nnU-Net cannot be used for training or inference. If this is not intended behavior, please read documentation/setting_up_paths.md for information on how to set this up.


## Iterative Point-Based Approach through Z-Axis

In [ ]:
## Load in data
sample_id = 1
mask_id = 1

source = "CVPR"
dataset = "LesionLocator"
timepoint = "Synthetic_Follow_Up"

images_path = dirs.PROCDATA / f"CVPR_LesionLocator/images/{timepoint}/LesionLocator_{sample_id:04d}"

img_data = sitk.ReadImage(images_path / "CT.nii.gz")
mask_data = sitk.ReadImage(images_path/ f"mask_{mask_id}.nii.gz")

img_data = sitk.DICOMOrient(img_data, 'LPS')
mask_data = sitk.DICOMOrient(mask_data, 'LPS')

img_arr = sitk.GetArrayFromImage(img_data) 
mask_arr = sitk.GetArrayFromImage(mask_data) 

In [ ]:
# Get the RERECIST line data 
max_area_slice = find_max_area_slice(mask_arr) # Get the slice with the largest pixel area 

# Find the center point of the RECIST line and the corners of the square centered around RECIST line midpoint 
# whose sides would be parallel and orthogonal to the RECIST line. 
negative_points, center_point, recist_points, pts_25_75, length = get_prompt_points(gt2D = mask_arr[max_area_slice], 
                                                                            spacing = img_arr.shape) 
RERECIST_SCRIB, BBOX_ROTATED, MIN_AX_PTS, PTS_25_75 = transform_prompt_points(negative_pts = negative_points, 
                                                                                  recist_pts = recist_points, 
                                                                                  pts_25_75 = pts_25_75, 
                                                                                  max_area_slice = max_area_slice, 
                                                                                  img_shape = mask_arr.shape)
recist_line = get_line_from_recist(recist_coords = recist_points, slice_number = max_area_slice, img_size = img_arr.shape)

# 2 Positive 4 Negative point prompts

In [9]:
## Initiate session and use the middle of the RERECIST line as the first input 
session = nnInteractiveInferenceSession(
    device=torch.device("cpu"),  # Set inference device
    use_torch_compile=False,  # Experimental: Not tested yet
    verbose=False,
    torch_n_threads=os.cpu_count(),  # Use available CPU cores
    do_autozoom=True,  # Enables AutoZoom for better patching
    use_pinned_memory=True,  # Optimizes GPU memory transfers
)

model_path = os.path.join(DOWNLOAD_DIR, MODEL_NAME)
session.initialize_from_trained_model_folder(model_path)

# Transform imaging to be compatible with nnInteractive 
img = img_arr[None].transpose(0, 2, 3, 1) # make sure that shape is transformed from (1, z, x, y) to (1, x, y, z)

# Validate input dimensions
if img.ndim != 4:
    raise ValueError("Input image must be 4D with shape (1, x, y, z)")

session.set_image(img)

# --- Define Output Buffer ---
target_tensor = torch.zeros(img.shape[1:], dtype=torch.uint8)  # Must be 3D (x, y, z)
session.set_target_buffer(target_tensor)

# Enter 25 and 75 percentile point along RECIST as positive prompts
session.add_point_interaction(PTS_25_75[0], include_interaction=True)
session.add_point_interaction(PTS_25_75[1], include_interaction=True)

# Enter negative points at rotated bounding box corners as negative prompts
for point in BBOX_ROTATED:
    session.add_point_interaction(point, include_interaction=False)

# Transform results back to z, x, y form 
results_xyz = session.target_buffer.clone()
results_zxy = results_xyz.cpu().numpy().transpose(2, 0, 1)

License reminder: The official nnInteractive checkpoint is licensed under Creative Commons Attribution Non Commercial Share Alike 4.0 (CC BY-NC-SA 4.0). See the license note in readme.md (# License).
Unable to locate trainer class nnInteractiveTrainer_stub in nnInteractive.trainer. Please place it there (in any .py file)!
Attempting to use default nnInteractiveTrainer_stub. If you encounter errors, this is where you need to look!
Added new point interaction: center 1, scale [[311, 173, 296]]


: 

In [37]:
# Save out prediction masks
out_path = Path(str(images_path).replace('images', 'predictions'))
out_path.mkdir(parents=True, exist_ok=True)

result_mask_image = sitk.GetImageFromArray(results_zxy)
result_mask_image.SetSpacing(mask_data.GetSpacing())
sitk.WriteImage(result_mask_image, out_path / f"pred_mask_{mask_id}.nii.gz")

In [17]:
## Visualize first input and evaluate performance 
# Evaluate performance
metric_eval1 = Evaluator() 
metric_dict1 = metric_eval1(preds = results_zxy, 
                          targets = mask_arr, 
                          spacing = mask_data.GetSpacing())

metric_df1 = pd.DataFrame(metric_dict1, index = [0])

print(metric_df1)

/home/bhkuser/bhklab/katy/bhklab-nninteractive/.pixi/envs/default/lib/python3.12/site-packages/monai/utils/deprecate_utils.py:221: FutureWarning: monai.metrics.utils get_mask_edges:always_return_as_numpy: Argument `always_return_as_numpy` has been deprecated since version 1.5.0. It will be removed in version 1.7.0. The option is removed and the return type will always be equal to the input type.
  warn_deprecated(argname, msg, warning_category)


   volume_dice  jaccard  hausdorff  surface_distance  surface_dice  \
0     0.881744   0.7885   1.659583          0.411187      0.999075   

   panoptic_quality  added_path_length  false_negative_volume  \
0               1.0                673                    240   

   false_negative_path_length  
0                         240  
